In [1]:
from grape import Graph
import numpy as np
import pandas as pd
import pickle

/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Notebook to generate embeddings for NP-KG version 2 

## Embedding methods
- TransE, LINE, HOPE, node2vec, deepwalk, SPINE (weighted), RotatE, HolE

## Graph types
- Multidigraph with multiple edges between nodes (aggregated as embedded features)
- Digraph with single hashed edge between nodes (aggregated prior to embedding)

## Steps for Experiment 1
- load both versions of graph
- split into train and test connected holdout sets (k fold possible for hashed graph)
- choose hyperparameters for embedding methods (default in experiment 1)
- generate embeddings for train graph
- save embeddings
- visualize embeddings (also: sum, hadamard, avg of embeddings for multigraph)
- edge prediction intrinsic evaluation - in separate notebook
- edge prediction extrinsic evaluation - in separate notebook

In [2]:
KG_PATH = '/home/sanya/np-kg/resources/knowledge_graphs/'
DIR_PATH = 'grape-embeddings/'

In [3]:
npkg_default = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v2.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE_v2.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [ ]:
npkg_default_und = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v2.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE_v2.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=False,
    verbose=True
)

In [ ]:
npkg_hashed = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v2.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v2.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [ ]:
npkg_hashed_und = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v2.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v2.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=False,
    verbose=True
)

In [ ]:
npkg_default = npkg_default.remove_disconnected_nodes()
npkg_default_und = npkg_default.remove_disconnected_nodes()

In [ ]:
npkg_hashed = npkg_default.remove_disconnected_nodes()
npkg_hashed_und = npkg_default.remove_disconnected_nodes()

In [ ]:
from grape import GraphVisualizer
#Node degree distribution. Node degrees are on the horizontal axis and node counts are on the vertical axis on a logarithmic scale.
GraphVisualizer(npkg_default).plot_node_degree_distribution()

In [ ]:
GraphVisualizer(npkg_hashed).plot_node_degree_distribution()

## Split into train and test

In [ ]:
train_default, test_default = npkg_default.connected_holdout(
    train_size = 0.8,
    random_state=42,
    include_all_edge_types=True,
    #edge_types = [optional list of strings if restricting holdout/evaluation to specific edge types]
    verbose=True
)

In [ ]:
train_hashed, test_hashed = npkg_hashed.connected_holdout(
    train_size = 0.8,
    random_state=42,
    include_all_edge_types=True,
    #edge_types = [optional list of strings if restricting holdout/evaluation to specific edge types]
    verbose=True
)

## Generate embeddings

### TransE - multigraph

In [ ]:
from grape.embedders import TransEEnsmallen

In [ ]:
#Defaults: embedding_size=100, epochs=100, relu_bias=1.0, learning_rate=0.05, learning_rate_decay=0.9, random_state=42
#Time taken on CPU: 8 minutes
transe_embed_model = TransEEnsmallen(
    embedding_size=100,
    epochs=100,
    random_state=42,
    verbose=True,
    enable_cache=True
)
transe_embedding = transe_embed_model.fit_transform(npkg_default)

In [ ]:
transe_embed_model.parameters()

{'random_state': 42,
 'embedding_size': 100,
 'relu_bias': 1.0,
 'epochs': 100,
 'learning_rate': 0.05,
 'learning_rate_decay': 0.9,
 'node_embedding_path': 'grape-embeddings/npkg-transE-node-embedding-default.txt',
 'verbose': True,
 'dtype': 'f32',
 'edge_type_embedding_path': 'grape-embeddings/npkg-transE-edge-type-embedding-default.txt'}

In [ ]:
##save embeddings (if no path provided above)
import pickle
transe_embedding_dict = transe_embedding.dump()
with open(DIR_PATH+'npkg_transE_embedding_dict_default.pickle', 'wb') as file_trans:
    pickle.dump(transe_embedding_dict, file_trans)

In [ ]:
#load embeddings
from grape import EmbeddingResult
with open(DIR_PATH+'npkg_transE_embedding_dict_default.pickle', 'rb') as file_i:
    transe_embedding_dict = pickle.load(file_i)
transe_embedding = EmbeddingResult.load(transe_embedding_dict)

In [ ]:
##take a look at the node embedding
#index of dataframe are the node URIs from the graph
transe_embedding.get_node_embedding_from_index(0)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
ensembl:ENST00000669649,0.133012,-0.039734,-0.017600,0.074753,0.089565,-0.129705,-0.061305,-0.112825,-0.035546,0.039217,...,0.091381,-0.055465,-0.049071,0.015248,0.085441,0.127182,-0.048844,-0.007128,0.117262,0.134768
uberon:0006566,-0.042953,0.119542,0.106486,-0.095515,-0.072714,0.052780,0.075805,0.058281,0.089060,-0.130521,...,-0.048847,0.095971,0.100481,-0.161149,-0.065478,-0.082184,0.092494,-0.150797,-0.061185,-0.062067
ensembl:ENST00000710852,0.129897,-0.055989,-0.048424,0.068339,0.081475,-0.126741,-0.086929,-0.112643,-0.063531,0.071894,...,0.110342,-0.075432,-0.068375,0.034195,0.100543,0.146299,-0.071475,0.030592,0.111794,0.122829
so:0001877,-0.032624,-0.112300,-0.091139,-0.074079,-0.072230,0.035402,-0.120046,0.045256,-0.112124,0.127538,...,0.154696,-0.133258,-0.120145,0.086671,0.140970,0.131788,-0.126846,0.084415,-0.045741,-0.062484
uberon:0002046,-0.043733,0.120863,0.109191,-0.096344,-0.073366,0.053372,0.077542,0.059337,0.091038,-0.131430,...,-0.049722,0.097291,0.101299,-0.162655,-0.066260,-0.083180,0.093742,-0.153220,-0.061787,-0.063820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
pr:p51984,-0.071030,-0.122624,-0.046186,-0.046063,-0.066687,0.062966,-0.114123,0.046304,-0.103504,0.140603,...,0.097197,-0.136954,-0.110243,0.113062,0.138561,0.285851,-0.115542,0.087326,-0.049446,-0.079660
pr:q3iz91,-0.063238,-0.118321,-0.114408,-0.095154,-0.072189,0.095164,-0.114518,0.090444,-0.091321,0.123619,...,0.117939,-0.113842,-0.124325,0.061563,0.129074,0.135218,-0.128681,0.108919,-0.072243,-0.073731
go:0034019,-0.132350,-0.073959,-0.074117,-0.136843,-0.109497,0.114793,-0.116247,0.115912,-0.080128,0.071386,...,0.108645,-0.084689,-0.093388,0.059807,0.095091,0.099150,-0.081326,0.062787,-0.104859,-0.107982
pr:p83965,-0.041061,-0.127165,-0.111282,-0.081521,-0.061617,0.068471,-0.141537,0.057336,-0.118830,0.099643,...,0.128686,-0.113616,-0.147926,0.095228,0.133988,0.160273,-0.103372,0.095524,-0.116441,-0.019295


In [ ]:
##take a look at the edge embedding (generated for the 4 different edge types in the graph)
transe_embedding.get_edge_type_embedding_from_index(0)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
ro:0001025,-6.636734,7.613238,8.224732,-7.443454,-8.477786,5.672297,9.893656,6.597908,9.296768,-5.454515,...,-9.752181,7.710365,9.926104,-6.169213,-10.322266,-5.666488,8.394701,-6.800806,-6.763617,-6.817780
ro:0001015,7.695096,-6.392290,-7.731688,8.045654,9.637430,-6.672614,-8.348529,-7.590189,-8.243292,4.813333,...,8.375585,-6.841699,-7.863305,5.166222,8.724146,5.584220,-7.568592,5.699937,8.164729,8.129503
http://www.w3.org/2000/01/rdf-schema#subclassof,-4.857944,-4.488316,-5.306038,-5.061757,-5.226358,4.625896,-5.259227,4.903893,-5.411017,3.888401,...,5.417539,-4.956500,-4.908848,4.176100,5.219866,2.498688,-5.245125,4.570869,-4.770346,-4.184330
bfo:0000050,-3.382227,2.835286,4.321743,-3.384536,-3.567644,3.448605,4.351221,3.640927,4.360774,2.665185,...,-4.290301,3.729779,3.589310,2.424194,-4.167218,-3.289349,3.833445,-2.947739,-3.393661,-3.561412
ro:0000056,6.205667,-5.067720,-5.573077,6.795090,-6.602760,-6.291440,7.374416,-6.419090,-5.992821,4.678592,...,-7.274004,-5.312018,6.786849,4.380886,-7.060073,5.181231,-5.605486,4.653585,6.386588,-5.182482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ro:0002509,-0.314047,-0.218692,-0.195120,0.088319,-0.049960,-0.076689,0.031821,0.195661,-0.197387,-0.142221,...,-0.037379,-0.132735,-0.183673,0.325302,-0.139012,0.058360,-0.026303,-0.169888,-0.277753,-0.001862
so#transcribed:from,-0.245226,-0.321950,-0.254210,-0.101990,-0.253767,-0.131768,-0.029788,0.154678,0.003670,0.315482,...,0.191094,-0.289768,-0.192100,-0.073771,-0.192126,-0.347457,-0.078338,0.048236,0.068350,0.047734
foodon:00001301,0.095848,0.106489,-0.277093,0.131017,-0.212231,0.102525,-0.211474,0.100451,0.130998,-0.058926,...,-0.024576,0.050765,0.061389,0.215195,0.026648,0.231535,0.024710,0.175974,-0.188626,0.147543
ncit:r81,-0.051298,-0.010116,-0.025805,0.086762,-0.147431,-0.040595,0.101275,-0.239957,-0.021895,-0.092163,...,-0.280338,0.132704,-0.147428,-0.070553,0.060339,-0.186369,0.235242,-0.110890,0.289624,-0.145919


### TransE - hashed graph

In [ ]:
#Defaults: embedding_size=100, epochs=100, relu_bias=1.0, learning_rate=0.05, learning_rate_decay=0.9, random_state=42
#Time taken on CPU: 8 minutes
transe_embed_model_hash = TransEEnsmallen(
    embedding_size=100,
    epochs=100,
    random_state=42,
    verbose=True,
    enable_cache=True
)
transe_embedding_hased = transe_embed_model_hash.fit_transform(npkg_hashed)

In [ ]:
##save embeddings (if no path provided above)
transe_embedding_dict = transe_embedding_hased.dump()
with open(DIR_PATH+'npkg_transE_embedding_dict_default_hashed.pickle', 'wb') as file_trans:
    pickle.dump(transe_embedding_dict, file_trans)

### Node2vec - Multigraph

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen, Node2VecGloVeEnsmallen, Node2VecCBOWEnsmallen

In [ ]:
##Hyperparameters: Return parameter (p): [0.25, 0.5, 1, 2, 4]; In-Out (q): [0.25, 0.5, 1, 2, 4]

In [ ]:
#Node2Vec CBOW
#Defaults: embedding_size=100, epochs=30, clipping_value=6, number_of_negative_samples=10, walk_length=128, iterations=10,
#window_size=5, return_weight (inverse of p)=0.25 (higher makes BFS), explore_weight=4 (inverse of q, higher gives DFS),
#max_neighbours=100, random_state=42, normalize_by_degree=False (change see notes)
node2vec_cbow_embed_model = Node2VecCBOWEnsmallen(
    embedding_size=100,
    epochs=100,
    clipping_value=6,
    number_of_negative_samples=10,
    walk_length=128,
    iterations=100,
    window_size=10,
    return_weight=0.25,
    explore_weight=4,
    max_neighbours=100,
    random_state=42,
    normalize_by_degree=True,
    verbose=True,
    enable_cache=True
)
node2vec_cbow_embedding_default = node2vec_cbow_embed_model.fit_transform(npkg_default)

In [ ]:
node2vec_cbow_embedding_default_dict = node2vec_cbow_embedding_default.dump()
with open(DIR_PATH+'npkg_node2vec_cbow_embedding_dict_default.pickle', 'wb') as file_n2v:
    pickle.dump(node2vec_cbow_embedding_default_dict.dump(), file_n2v)

In [ ]:
#Node2Vec Skipgram
#Defaults: embedding_size=100, epochs=30, clipping_value=6, number_of_negative_samples=10, walk_length=128, iterations=10,
#window_size=5, return_weight (inverse of p)=0.25 (higher makes BFS), explore_weight=4 (inverse of q, higher gives DFS),
#max_neighbours=100, random_state=42, normalize_by_degree=False (change see notes)
node2vec_skipgram_embed_model = Node2VecSkipGramEnsmallen(
    embedding_size=100,
    epochs=100,
    clipping_value=6,
    number_of_negative_samples=10,
    walk_length=128,
    iterations=100,
    window_size=10,
    return_weight=0.25,
    explore_weight=4,
    max_neighbours=100,
    random_state=42,
    normalize_by_degree=True,
    verbose=True,
    enable_cache=True
)
node2vec_skipgram_embedding_default = node2vec_skipgram_embed_model.fit_transform(npkg_default)

In [ ]:
##save embedding
node2vec_skipgram_embedding_default_dict = node2vec_skipgram_embedding_default.dump()
with open(DIR_PATH+'npkg_node2vec_skipgram_embedding_dict_default.pickle', 'wb') as file_n2v:
    pickle.dump(node2vec_skipgram_embedding_default_dict.dump(), file_n2v)

### Node2vec - hashed graph

In [ ]:
##Node2vec CBOW
node2vec_cbow_embed_model_hashed = Node2VecCBOWEnsmallen(
    embedding_size=100,
    epochs=100,
    clipping_value=6,
    number_of_negative_samples=10,
    walk_length=128,
    iterations=100,
    window_size=10,
    return_weight=0.25,
    explore_weight=4,
    max_neighbours=100,
    random_state=42,
    normalize_by_degree=True,
    verbose=True,
    enable_cache=True
)
node2vec_cbow_embedding_hashed = node2vec_cbow_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
##save embedding
node2vec_cbow_embedding_hashed_dict = node2vec_cbow_embedding_hashed.dump()
with open(DIR_PATH+'npkg_node2vec_cbow_embedding_dict_default_hashed.pickle', 'wb') as file_n2v:
    pickle.dump(node2vec_cbow_embedding_hashed_dict.dump(), file_n2v)

In [ ]:
##Node2vec skipgram
node2vec_skipgram_embed_model_hashed = Node2VecSkipGramEnsmallen(
    embedding_size=100,
    epochs=100,
    clipping_value=6,
    number_of_negative_samples=10,
    walk_length=128,
    iterations=100,
    window_size=10,
    return_weight=0.25,
    explore_weight=4,
    max_neighbours=100,
    random_state=42,
    normalize_by_degree=True,
    verbose=True,
    enable_cache=True
)
node2vec_skipgram_embedding_hashed = node2vec_skipgram_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
##save embedding
node2vec_skipgram_embedding_hashed_dict = node2vec_skipgram_embedding_hashed.dump()
with open(DIR_PATH+'npkg_node2vec_skipgram_embedding_dict_default_hashed.pickle', 'wb') as file_n2v:
    pickle.dump(node2vec_skipgram_embedding_hashed_dict.dump(), file_n2v)

### DeepWalk

In [ ]:
from grape.embedders import DeepWalkCBOWEnsmallen, DeepWalkSkipGramEnsmallen

In [ ]:
##DeepWalk CBOW
##Defaults: embedding_size=100, epochs=30, clipping_value=6, number_of_negative_samples=10, walk_length=128, iterations=10,
##window_size=5, random_state=42, normalize_by_degree=False (change see notes), max_neighbours=100, learning_rate=0.01,
#learning_rate_decay=0.9
deepwalk_cbow_embed_model = DeepWalkCBOWEnsmallen(
    embedding_size=100,
    epochs=100,
    clipping_value=6,
    number_of_negative_samples=10,
    walk_length=128,
    iterations=100,
    window_size=10,
    random_state=42,
    normalize_by_degree=True,
    max_neighbours=100,
    learning_rate=0.01,
    learning_rate_decay=0.9,
    verbose=True,
    enable_cache=True
)
deepwalk_cbow_embedding_default = deepwalk_cbow_embed_model.fit_transform(npkg_default)

In [ ]:
##save 
deepwalk_cbow_embedding_default_dict = deepwalk_cbow_embedding_default.dump()
with open(DIR_PATH+'npkg_deepwalk_cbow_embedding_dict_default.pickle', 'wb') as file_dw:
    pickle.dump(deepwalk_cbow_embedding_default_dict.dump(), file_dw)

In [ ]:
##DeepWalk SkipGram
deepwalk_skipgram_embed_model = DeepWalkSkipGramEnsmallen(
    embedding_size=100,
    epochs=100,
    clipping_value=6,
    number_of_negative_samples=10,
    walk_length=128,
    iterations=100,
    window_size=10,
    random_state=42,
    normalize_by_degree=True,
    max_neighbours=100,
    learning_rate=0.01,
    learning_rate_decay=0.9,
    verbose=True,
    enable_cache=True
)
deepwalk_skipgram_embedding_default = deepwalk_skipgram_embed_model.fit_transform(npkg_default)

In [ ]:
##save
deepwalk_skipgram_embedding_default_dict = deepwalk_skipgram_embedding_default.dump()
with open(DIR_PATH+'npkg_deepwalk_skipgram_embedding_dict_default.pickle', 'wb') as file_dw:
    pickle.dump(deepwalk_skipgram_embedding_default_dict.dump(), file_dw)

### DeepWalk - Hashed Graph

In [ ]:
#DeepWalk CBOW
deepwalk_cbow_embed_model_hashed = DeepWalkCBOWEnsmallen(
    embedding_size=100,
    epochs=100,
    clipping_value=6,
    number_of_negative_samples=10,
    walk_length=128,
    iterations=100,
    window_size=10,
    random_state=42,
    normalize_by_degree=True,
    max_neighbours=100,
    learning_rate=0.01,
    learning_rate_decay=0.9,
    verbose=True,
    enable_cache=True
)
deepwalk_cbow_embedding_hashed = deepwalk_cbow_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
##save
deepwalk_cbow_embedding_hashed_dict = deepwalk_cbow_embedding_hashed.dump()
with open(DIR_PATH+'npkg_deepwalk_cbow_embedding_dict_default_hashed.pickle', 'wb') as file_dw:
    pickle.dump(deepwalk_cbow_embedding_hashed_dict.dump(), file_dw)

In [ ]:
##DeepWalk SkipGram
deepwalk_skipgram_embed_model_hashed = DeepWalkSkipGramEnsmallen(
    embedding_size=100,
    epochs=100,
    clipping_value=6,
    number_of_negative_samples=10,
    walk_length=128,
    iterations=100,
    window_size=10,
    random_state=42,
    normalize_by_degree=True,
    max_neighbours=100,
    learning_rate=0.01,
    learning_rate_decay=0.9,
    verbose=True,
    enable_cache=True
)
deepwalk_skipgram_embedding_hashed = deepwalk_skipgram_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
##save
deepwalk_skipgram_embedding_hashed_dict = deepwalk_skipgram_embedding_hashed.dump()

### LINE - Multigraph

In [ ]:
from grape.embedders import FirstOrderLINEEnsmallen, SecondOrderLINEEnsmallen

In [ ]:
##LINE First Order
#Default: embedding_size=100, epochs=100, learning_rate=0.05, learning_rate_decay=0.9
first_line_embed_model = FirstOrderLINEEnsmallen(
    embedding_size=100,
    epochs=100,
    learning_rate=0.05,
    learning_rate_decay=0.9,
    random_state=42
    verbose=True,
    enable_cache=True
)
first_line_embedding_default = first_line_embed_model.fit_transform(npkg_default)

In [ ]:
##save
first_line_embedding_default_dict = first_line_embedding_default.dump()
with open(DIR_PATH+'npkg_first_line_embedding_dict_default.pickle', 'wb') as file_line:
    pickle.dump(first_line_embedding_default_dict.dump(), file_line)

In [ ]:
##LINE Second Order
#Default: same as first order LINE
second_line_embed_model = SecondOrderLINEEnsmallen(
    embedding_size=100,
    epochs=100,
    learning_rate=0.05,
    learning_rate_decay=0.9,
    random_state=42
    verbose=True,
    enable_cache=True
)
second_line_embedding_default = second_line_embed_model.fit_transform(npkg_default)

In [ ]:
##save
second_line_embedding_default_dict = second_line_embedding_default.dump()
with open(DIR_PATH+'npkg_second_line_embedding_dict_default.pickle', 'wb') as file_line:
    pickle.dump(second_line_embedding_default_dict.dump(), file_line)

### LINE - Hashed Graph

In [ ]:
##LINE First Order
first_line_embed_model_hashed = FirstOrderLINEEnsmallen(
    embedding_size=100,
    epochs=100,
    learning_rate=0.05,
    learning_rate_decay=0.9,
    random_state=42
    verbose=True,
    enable_cache=True
)
first_line_embedding_hashed = first_line_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
#save
first_line_embedding_hashed_dict = first_line_embedding_hashed.dump()
with open(DIR_PATH+'npkg_first_line_embedding_dict_default_hashed.pickle', 'wb') as file_line:
    pickle.dump(first_line_embedding_hashed_dict.dump(), file_line)

In [ ]:
##LINE Second Order
second_line_embed_model_hashed = SecondOrderLINEEnsmallen(
    embedding_size=100,
    epochs=100,
    learning_rate=0.05,
    learning_rate_decay=0.9,
    random_state=42
    verbose=True,
    enable_cache=True
)
second_line_embedding_hashed = second_line_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
#save
second_line_embedding_hashed_dict = second_line_embedding_hashed.dump()
with open(DIR_PATH+'npkg_second_line_embedding_dict_default_hashed.pickle', 'wb') as file_line:
    pickle.dump(second_line_embedding_hashed_dict.dump(), file_line)

### HOPE - multigraph

In [ ]:
from grape.embedders import HOPEEnsmallen

In [ ]:
##HOPE
#Defaults: embedding_size=100, metric="Neighbours Intersection size", verbose=False
hope_embed_model = HOPEEnsmallen(
    embedding_size=100,
    metric="Neighbours Intersection size",
    verbose=True,
    enable_cache=True
)
hope_embedding_default = hope_embed_model.fit_transform(npkg_default)

In [ ]:
#save
hope_embedding_default_dict = hope_embedding_default.dump()
with open(DIR_PATH+'npkg_hope_embedding_dict_default.pickle', 'wb') as file_hope:
    pickle.dump(hope_embedding_default_dict.dump(), file_hope)

### HOPE - hashed graph

In [ ]:
##HOPE
hope_embed_model_hashed = HOPEEnsmallen(
    embedding_size=100,
    metric="Neighbours Intersection size",
    verbose=True,
    enable_cache=True
)
hope_embedding_hashed = hope_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
##save
hope_embedding_hashed_dict = hope_embedding_hashed.dump()

### SPINE - Multigraph

In [ ]:
from grape.embedders import WeightedSPINE, DegreeSPINE

In [ ]:
##Weighted SPINE
#Defaults: embedding_size=100, use_edge_weights_as_probabilities=False, verbose=False
spine_weighted_embed_model = WeightedSPINE(
    embedding_size=100,
    use_edge_weights_as_probabilities=True,
    verbose=True,
    enable_cache=True
)
spine_weighted_embedding_default = spine_weighted_embed_model.fit_transform(npkg_default)

In [ ]:
##save
spine_weighted_embedding_default_dict = spine_weighted_embedding_default.dump()
with open(DIR_PATH+'npkg_spine_weighted_embedding_dict_default.pickle', 'wb') as file_spine:
    pickle.dump(spine_weighted_embedding_default_dict.dump(), file_spine)

In [ ]:
##Degree SPINE
#Defaults: embedding_size=100, max_depth=None (max depth of shortest path), verbose=False
spine_degree_embed_model = DegreeSPINE(
    embedding_size=100,
    max_depth=None,
    verbose=True,
    enable_cache=True
)
spine_degree_embedding_default = spine_degree_embed_model.fit_transform(npkg_default)

In [ ]:
#save
spine_degree_embedding_default_dict = spine_degree_embedding_default.dump()
with open(DIR_PATH+'npkg_spine_degree_embedding_dict_default.pickle', 'wb') as file_spine:
    pickle.dump(spine_degree_embedding_default_dict.dump(), file_spine)

### SPINE - Hashed graph

In [ ]:
##weighted SPINE
spine_weighted_embed_model_hashed = WeightedSPINE(
    embedding_size=100,
    use_edge_weights_as_probabilities=True,
    verbose=True,
    enable_cache=True
)
spine_weighted_embedding_hashed = spine_weighted_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
##save
spine_weighted_embedding_hashed_dict = spine_weighted_embedding_hashed.dump()
with open(DIR_PATH+'npkg_spine_weighted_embedding_dict_default_hashed.pickle', 'wb') as file_spine:
    pickle.dump(spine_weighted_embedding_hashed_dict.dump(), file_spine)

In [ ]:
##Degree SPINE
spine_degree_embed_model_hashed = DegreeSPINE(
    embedding_size=100,
    max_depth=None,
    verbose=True,
    enable_cache=True
)
spine_degree_embedding_hashed = spine_degree_embed_model_hashed.fit_transform(npkg_hashed)

In [ ]:
##save
spine_degree_embedding_hashed_dict = spine_degree_embedding_hashed.dump()
with open(DIR_PATH+'npkg_spine_degree_embedding_dict_default_hashed.pickle', 'wb') as file_spine:
    pickle.dump(spine_degree_embedding_hashed_dict.dump(), file_spine)

### Confirm model params

In [ ]:
##For all models, check: can_use_edge_weights, is_using_edge_weights, can_use_edge_types, is_using_edge_types, can_use_node_types, is_using_node_types


## Visualization

In [ ]:
%%time
from grape import GraphVisualizer

vis = GraphVisualizer(
    graph=test,
    support=train,
    n_components=4,
    edge_embedding_method="Hadamard",
    rotate=True,
    verbose=True,
    # Automatically, since LINE learns a cosine, the visualization tool
    # would dispatch a Cosine-distance based TSNE. This would use the sklearn
    # implementation, which is terribly slow. Therefore, we force it to use the Euclidean distance
    # and therefore the Multicore TSNE implementation (when available).
    decomposition_kwargs=dict(metric="euclidean")
)

In [ ]:
GraphVisualizer(graph).fit_and_plot_all(degree_normalized_deepwalk_embedding)

In [ ]:
vis.fit_negative_and_positive_edges(second_order_embedding)

In [ ]:
vis.plot_positive_and_negative_edges()


In [ ]:
from grape import GraphVisualizer

vis = GraphVisualizer(
    graph=graph,
    n_components=4,
    edge_embedding_method="Hadamard",
    rotate=True,
    verbose=True,
    # Automatically, since GloVe learns a cosine, the visualization tool
    # would dispatch a Cosine-distance based TSNE. This would use the sklearn
    # implementation, which is terribly slow. Therefore, we force it to use the Euclidean distance
    # and therefore the Multicore TSNE implementation (when available).
    decomposition_kwargs=dict(metric="euclidean")
)
vis.fit_nodes(deepwalk_embedding)
vis.plot_node_types()
